In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
def generate_data(n = 100):
    rng1 = np.random.default_rng(seed=42)
    y = (rng1.random(n) > 0.5).astype(int)
    rng2 = np.random.default_rng(seed=665)
    y_pred_proba = rng2.random(n)
    y_pred = (y_pred_proba > 0.5).astype(int)
    return y, y_pred_proba, y_pred
    

In [4]:
y, y_pred_proba, y_pred = generate_data()

**1. Accuracy** 

What proportion of overall predictions are correct?

In [5]:
def accuracy(y, y_pred):
    return np.mean(y==y_pred)   # np.sum(y==y_pred)/len(y)

accuracy(y, y_pred)

np.float64(0.58)

In [6]:
# sklearn

from sklearn.metrics import accuracy_score

accuracy_score(y, y_pred)

0.58

**2. Confusion Matrix**

<img src = 'metadata/confusion-matrix.png' height = 300  width = 400>

**3. Precision**

What propertion of positive predictions are correct?

True Positive (Tp) = positive predictions which are actually positive

False Positive (Fp) = positive predictions which are actually negative

Precision = Tp/(Tp+Fp)

In [7]:
def precision(y, y_pred):
    # tp = np.sum((y==1) & (y_pred==1
    # fp = np.sum((y==0) & (y_pred==1))
    # return tp/(tp+fp)

    return np.sum((y==1) & (y_pred==1))/np.sum(y_pred == 1)

In [8]:
precision(y, y_pred)

np.float64(0.5555555555555556)

In [9]:
# sklearn

from sklearn.metrics import precision_score

precision_score(y, y_pred)

0.5555555555555556

**4. Recall**

What proportion of actually positive samples were predicted as positive

False Negative (Fn) = predicted negative but is actually positive

Recall = Tp/(Tp+Fn)

In [10]:
def recall(y, y_pred):
    # tp = np.sum((y==1) & (y_pred == 1))
    # fn = np.sum((y==1) & (y_pred == 0))
    # return tp/(tp+fn)

    return np.sum((y==1) & (y_pred==1))/np.sum(y==1)

In [11]:
recall(y, y_pred)

np.float64(0.5319148936170213)

In [12]:
# sklearn

from sklearn.metrics import recall_score

recall_score(y, y_pred)

0.5319148936170213

**5. F1 Score**

It is the harmonic mean of precision and recall

In [13]:
def f1(y, y_pred):
    p = precision(y, y_pred)
    r = recall(y, y_pred)
    return 2/((1/p) + (1/r))

In [14]:
f1(y, y_pred)

np.float64(0.5434782608695653)

In [15]:
# sklearn

from sklearn.metrics import f1_score

f1_score(y, y_pred)

0.5434782608695652

**6. Precision vs Recall**

Precision and Recall are more insightful than Accuracy for inbalanced data and offer a complete picture of model performance. While we strive to improve both Precision and Recall of a classifier, in a real world system often efforts to improve Precision (by increasing the classification threshold such that the model only predicts positive when it is highly confident) can decrease Recall (the model makes very less positive predictions so False Negatives increase) and vice. versa. So how to make the trade-off? Than depends entirely on the business cost of False Positives (Precision) vs False Negatives (recall). For example: if the business objective is to build fraud detection in credit card transaction, then for this system the cost of  False Negatives (the system fails to flag actual fraud instances) is much higher than False Positives (geniunine transactions flagged as fraud, business cost - disgruntled customer but can be resolved through customer support). Thus in production such a system should always err on the side of improving Recall (reducing False Negative) over Precision (False Positive) by for instance reducing the classification threshold.

**7. Threshold-Independent Ranking Metrics**

These evaluate the model's internal probability scores, not its final threshold-dependent 0/1 guess, and are thus very insightful

**7.1. ROC-AUC (Area Under the Receiver Operating Characteristic Curve)**

Plots TPR (Recall) (y-axis) against FPR (False Positive Rate) (x-axis) at various thresholds. .

Interpretation: The probability that the model will rank a randomly chosen positive instance higher than a randomly chosen negative instance (called pairwise concordance)

Use When: Classes are roughly balanced and we care about the general ranking ability of the model.

Calculated using Mann-Whitney U test (a non-parametric statistical test) AUC = Concordant Pairs/Total Pairs

Sometimes we will use The Gini Coefficient which scales the AUC from a range of [0.5, 1.0] to [0, 1] Gini = 2*AUC - 1

In [30]:
def roc_auc(y, y_pred_proba):

    # 1. Sort Predictions:
    sorted_idx = np.argsort(y_pred_proba)
    sorted_labels = y[sorted_idx]

    # 2. Get total positives and negatives
    m = np.sum(y)
    n = len(y) - m

    # 3. get the ranks
    pos_ranks = np.where(sorted_labels == 1)[0] + 1

    # 4. calculate the ranks for positive instances
    sum_pos_ranks = np.sum(pos_ranks)

    # 5. Apply the Mann-Whitney U formula
    auc = (sum_pos_ranks - (m * (m+1) / 2))/(m * n)

    return auc

In [31]:
roc_auc(y, y_pred_proba)

np.float64(0.5483741469289442)

In [32]:
# sklearn

from sklearn.metrics import roc_auc_score
roc_auc_score(y, y_pred_proba)

0.5483741469289443

**7.2. PR-AUC**

Plots Precision (y-axis) against Recall (x-ais) at various thresholds.

Use when there is a severe class imbalance. This is far more sesitive to the minority class than ROC-AUC and penalises the models generating many false positives.



In [33]:
# sklearn

from sklearn.metrics import average_precision_score

average_precision_score(y, y_pred_proba)

0.5123102017290551

**7.3. Expected Pairwise Rank (or R-pair)**

Paiwise Ranking Loss calculates the total number of inversions in prediction. Instead of using area under a curve (AUC), it uses Spearman's Rank Correlation between predicted scores and actual labels. Penalises the models heavily if they misorder the very top of the list.

Used in recommendation systems where getting the the top result right is more important than the rest.

For Binary Classification it collapses to being the AUC-ROC

**8. The Kolmogorov-Smirnov Statistic**

It compares the cumulative distribution functions (CDFs) of the predicted score for the Positive and Negative Classes. The KS statistic is the maximum vertical distance between these two CDF curves. It is a measure of the peak discriminative power at a single best threshold.

Mathematically it is max(TPR - FDR) = the biggest gap between your cumulative positive and negative distributions

Interpretation: If KS is found to be 0.40, it means that at the optimal split point, the model achieves a 40%  separation between the two classes 

**9. Brier Score**

The mean square error of probabilities. (1/N) * Σ (p_i - y_i)². Unlike Log-Loss, Brier is bounded between 0 and 1. 

Statistical Decomposition of Brier Score (Murphy's Theorem) 
- Reliability (~Calibration)
- Resolution (~Discrimination)
- Uncertainty (~Noise)

